# TF-IDF

Based on Term Frequency/Inverted Index last time

The text files are placed in folder './iitfidf', i.e. we have three files in the folder: 1.txt, 2.txt, 3.txt

The wholeTextFiles will read the files, and output a tuple (filename, fulltext)

data (i.e., wholeTextFiles) -> (filename, fulltext)

Here we calculate the tf "Term Frequency":
**tf = 1+log10(tf)**

In [1]:
import re
def removeSomeTexts(text):
    return re.sub(r"[^a-z-'\s]",'', text) # only keep: a-z, - and ' \s

In [2]:
from pyspark import SparkConf, SparkContext
import os, re
from pyspark.sql.session import SparkSession

spark = SparkSession.builder.master('local').appName('MyTf-Idf').getOrCreate()
sc = spark.sparkContext
#sc = SparkContext.getOrCreate(SparkConf())

#sc._conf.set('spark.executor.memory', '8g')
#sc._conf.set('spark.driver.memory', '8g')
#sc._conf.set('spark.driver.maxResultsSize', '0')

#data (i.e., wholeTextFiles) -> (filename, fulltext)
data = sc.wholeTextFiles('data/iitfidf')

numFiles = data.count()

# ((filename, token), TF)
wordcount = data.flatMap(lambda x: [((os.path.basename(x[0]), removeSomeTexts(i.lower())), 1) for i in re.split('\W+', x[1])]) \
        .reduceByKey(lambda a, b: a + b)
wordcount.take(10)   

[(('1.txt', 'knitting'), 1),
 (('1.txt', 'is'), 1),
 (('1.txt', 'my'), 3),
 (('1.txt', 'hobby'), 1),
 (('1.txt', 'and'), 2),
 (('1.txt', 'passion'), 1),
 (('1.txt', 'i'), 1),
 (('1.txt', 'love'), 2),
 (('1.txt', 'dogs'), 1),
 (('1.txt', 'cats'), 1)]

In [3]:
# ((filename, token), TF) => ((token, (filename, TF)) 
tf1 = wordcount.map(lambda x: (x[0][1],(x[0][0],x[1])))
tf1.take(10)

[('knitting', ('1.txt', 1)),
 ('is', ('1.txt', 1)),
 ('my', ('1.txt', 3)),
 ('hobby', ('1.txt', 1)),
 ('and', ('1.txt', 2)),
 ('passion', ('1.txt', 1)),
 ('i', ('1.txt', 1)),
 ('love', ('1.txt', 2)),
 ('dogs', ('1.txt', 1)),
 ('cats', ('1.txt', 1))]

In [4]:
# tf = 1+log10(tf)
import math
tf = tf1.map(lambda x: (x[0],(x[1][0], 1+math.log10((x[1][1]) ))))

tf.take(10)

[('knitting', ('1.txt', 1.0)),
 ('is', ('1.txt', 1.0)),
 ('my', ('1.txt', 1.4771212547196624)),
 ('hobby', ('1.txt', 1.0)),
 ('and', ('1.txt', 1.3010299956639813)),
 ('passion', ('1.txt', 1.0)),
 ('i', ('1.txt', 1.0)),
 ('love', ('1.txt', 1.3010299956639813)),
 ('dogs', ('1.txt', 1.0)),
 ('cats', ('1.txt', 1.0))]

## Computing Inverse Document Frequency (IDF)

The IDF score tells us how important is a particular word in the whole corpus. For instance, if a word is appearing in each document then the IDF score will be zero for that.

We know that the total number of documents for our case is 3. So, our next step will be to compute how many documents contain a particular token w. Further, we can use this information to calculate the IDF scores for all the tokens in the corpus.

We will be chaining the previous map/reduce computation to calculate IDF in this section. After the reduced step in the previous section, we obtained key/value pairs for respective tokens along with its term frequency.

In the first step, we will map the previous key/value pair to a new key/value pair. Here, the key will the token and its value will be the document id TF for that token along with a counter of 1. This one here indicates the presence of a word in document id associated with that.

 ((token, (filename, TF)) -> (token, 1)

Then, we will reduce by key to obtain the count of documents containing a particular token w (wordcount).

(token, 1) -> (token, 1+1+1+...)

In the last step, since we have the number of documents containing each token w, we will just map this final output with the logarithmic transformation to compute the IDF score.

(token, doc-freq-containing-token) -> (token, IDF)

In [5]:
# ((token, filename), TF) -> (token, 1)
idf2 = tf1.map(lambda x: (x[0], 1))
idf2.take(10)

[('knitting', 1),
 ('is', 1),
 ('my', 1),
 ('hobby', 1),
 ('and', 1),
 ('passion', 1),
 ('i', 1),
 ('love', 1),
 ('dogs', 1),
 ('cats', 1)]

In [6]:
# (token, 1) -> (token, 1+1+1+...)
idf3 = idf2.reduceByKey(lambda x, y: x + y)
idf3.take(10)

[('knitting', 1),
 ('is', 3),
 ('my', 1),
 ('hobby', 1),
 ('and', 3),
 ('passion', 1),
 ('i', 1),
 ('love', 1),
 ('dogs', 1),
 ('cats', 1)]

In [7]:
# (token, doc-freq-containing-token) -> (token, IDF)
idf = idf3.map(lambda x: (x[0], math.log10(numFiles / (x[1]))))
idf.take(10)

[('knitting', 0.47712125471966244),
 ('is', 0.0),
 ('my', 0.47712125471966244),
 ('hobby', 0.47712125471966244),
 ('and', 0.0),
 ('passion', 0.47712125471966244),
 ('i', 0.47712125471966244),
 ('love', 0.47712125471966244),
 ('dogs', 0.47712125471966244),
 ('cats', 0.47712125471966244)]

## Computing TF-IDF

Now we have two RDDs one having the document id along with its document term frequency for each token. Another RDD having IDF scores for each token.

First, we will perform an inner join to assign each token with a document id, TF, and IDF score.

Second, we will map this function to multiply TF and IDF values of each token associated with respective document id.

Third, we sort the key

New Key/Value pair -> (filename, (token, TF-IDF))

In [8]:
tfidf1 = tf.join(idf)
tfidf1.take(10)

[('is', (('1.txt', 1.0), 0.0)),
 ('is', (('2.txt', 1.0), 0.0)),
 ('is', (('3.txt', 1.0), 0.0)),
 ('passion', (('1.txt', 1.0), 0.47712125471966244)),
 ('i', (('1.txt', 1.0), 0.47712125471966244)),
 ('love', (('1.txt', 1.3010299956639813), 0.47712125471966244)),
 ('dogs', (('1.txt', 1.0), 0.47712125471966244)),
 ('cats', (('1.txt', 1.0), 0.47712125471966244)),
 ('these', (('1.txt', 1.0), 0.47712125471966244)),
 ('are', (('1.txt', 1.0), 0.47712125471966244))]

In [9]:
tfidf = tfidf1.map(lambda x: (x[1][0][0], (x[0], x[1][0][1] * x[1][1]))).sortByKey()
tfidf.take(10)

[('1.txt', ('is', 0.0)),
 ('1.txt', ('passion', 0.47712125471966244)),
 ('1.txt', ('i', 0.47712125471966244)),
 ('1.txt', ('love', 0.6207490639591157)),
 ('1.txt', ('dogs', 0.47712125471966244)),
 ('1.txt', ('cats', 0.47712125471966244)),
 ('1.txt', ('these', 0.47712125471966244)),
 ('1.txt', ('are', 0.47712125471966244)),
 ('1.txt', ('knitting', 0.47712125471966244)),
 ('1.txt', ('my', 0.7047659464249274))]

In [10]:
# create an inverted index 
index = tfidf.map(lambda x:(x[1][0], (x[0], x[1][1]))).filter(lambda x: x[0]!='')
index.take(10)

[('is', ('1.txt', 0.0)),
 ('passion', ('1.txt', 0.47712125471966244)),
 ('i', ('1.txt', 0.47712125471966244)),
 ('love', ('1.txt', 0.6207490639591157)),
 ('dogs', ('1.txt', 0.47712125471966244)),
 ('cats', ('1.txt', 0.47712125471966244)),
 ('these', ('1.txt', 0.47712125471966244)),
 ('are', ('1.txt', 0.47712125471966244)),
 ('knitting', ('1.txt', 0.47712125471966244)),
 ('my', ('1.txt', 0.7047659464249274))]

In [11]:
index = index.reduceByKey(lambda a,b:a+b).sortByKey()
index.take(100)

[('a', ('2.txt', 0.22910001000567795, '3.txt', 0.17609125905568124)),
 ('all', ('2.txt', 0.47712125471966244)),
 ('alliance', ('2.txt', 0.47712125471966244)),
 ('and', ('1.txt', 0.0, '2.txt', 0.0, '3.txt', 0.0)),
 ('antagonisms', ('3.txt', 0.6207490639591157)),
 ('are', ('1.txt', 0.47712125471966244)),
 ('as', ('3.txt', 0.47712125471966244)),
 ('away', ('3.txt', 0.47712125471966244)),
 ('bourgeois', ('3.txt', 0.47712125471966244)),
 ('bourgeoisie', ('3.txt', 0.6207490639591157)),
 ('but', ('3.txt', 0.47712125471966244)),
 ('camps', ('3.txt', 0.47712125471966244)),
 ('cats', ('1.txt', 0.47712125471966244)),
 ('class', ('3.txt', 0.6207490639591157)),
 ('classes', ('3.txt', 0.6207490639591157)),
 ('communism', ('2.txt', 0.47712125471966244)),
 ('conditions', ('3.txt', 0.47712125471966244)),
 ('czar', ('2.txt', 0.47712125471966244)),
 ('directly', ('3.txt', 0.47712125471966244)),
 ('distinctive', ('3.txt', 0.47712125471966244)),
 ('dogs', ('1.txt', 0.47712125471966244)),
 ('done', ('3.txt'

In [ ]:
# sort tfidf value in reverse order for each word
# from ['3.txt', 0.17609125905568124', 2.txt', 0.22910001000567795, ]
# to ['2.txt', 0.22910001000567795, '3.txt', 0.17609125905568124]

import operator
def InvSort(x):
    if(len(x)<=2):
        return x
    # implement your code here
    #hint: you can use zip and dict to build a dictory from x
    #  use sorted to sort the dictory in value in reverse order
    # build a list from the dictory
    return x
        
index2 = index.map(lambda x: (x[0], InvSort(x[1])))
index2.take(100)

In [ ]:
# transform from a list to a tuple
index2 = index.map(lambda x: (x[0], tuple(x[1])))
index2.take(100)

In [ ]:
#save tfidf index to a text folder
# make sure there is no the following folder yet, otherwise, you need delete it first
index2.saveAsTextFile('data/tfidf_txt')